In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
# ---------- Prompt templates ----------
TEMPLATES = {
    1: """Prompt:
{prompt}

Response A:
{a}

Response B:
{b}

Question: Which response is better? (A, B, or Tie)""",

    2: """You are a judge comparing two AI responses to the same user prompt.
Read carefully and decide which is better.

Prompt:
{prompt}

Answer A:
{a}

Answer B:
{b}

Which one is preferred? Options: [A] [B] [Tie]""",

    3: """Prompt from user:
{prompt}

Response from Model A:
{a}

Response from Model B:
{b}

As a judge, choose the response that is more helpful, accurate, and clear.
Your decision: (A, B, or Tie)""",

    5: """Task: Compare two responses to the same prompt.

Prompt:
{prompt}

--- Response A ---
{a}

--- Response B ---
{b}

Task: Select the better response (A, B, or Tie). Do not assume A is better just because it comes first."""
}


In [ ]:
import torch
from torch.utils.data import Dataset
import torch.nn as nn
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.metrics import accuracy_score, f1_score


from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding,
    set_seed,
)

# -------------------------
# Config
# -------------------------
SEED = 42
set_seed(SEED)

MODEL_NAME = "/kaggle/input/distilbertdistilbert-base-uncased/transformers/default/1"  # small, fast
NUM_LABELS = 3
MAX_LENGTH = 384

EPOCHS = 2
LR = 2e-5
OUTPUT_DIR = "/kaggle/working/model"
BATCH_SIZE = 16

# Paths (change if your competition paths differ)
TRAIN_CSV = "/kaggle/input/llm-classification-finetuning/train.csv"      # path may differ; adjust
TEST_CSV  = "/kaggle/input/llm-classification-finetuning/test.csv"       # path may differ; adjust

In [ ]:

# -------------------------
# Load Data
# -------------------------
train_df = pd.read_csv(TRAIN_CSV)
test_df  = pd.read_csv(TEST_CSV)
# Targets: one-hot -> label ids
y = train_df[["winner_model_a", "winner_model_b", "winner_tie"]].values.argmax(axis=1)

# -------------------------
# Tokenizer & Dataset
# -------------------------
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


In [ ]:
class PairDataset(Dataset):
    def __init__(self, df, labels=None):
        self.df = df.reset_index(drop=True)
        self.labels = labels

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        r = self.df.iloc[idx]
        text = format_input(r["prompt"], r["response_a"], r["response_b"])
        enc = tokenizer(text, truncation=True, max_length=MAX_LENGTH, return_tensors="pt")
        item = {k: v.squeeze(0) for k,v in enc.items()}
        if self.labels is not None:
            item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

train_ds = PairDataset(train_df, y)
test_ds  = PairDataset(test_df, None)
collator = DataCollatorWithPadding(tokenizer)

In [ ]:
def format_input(prompt, a, b, tid):
    tpl = TEMPLATES[tid]
    return tpl.format(prompt=prompt, a=a, b=b)
    # return f"Prompt:\n{prompt}\n\nResponse A:\n{a}\n\nResponse B:\n{b}"

In [ ]:
EPOCHS = 1
print(torch.cuda.is_available())

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=NUM_LABELS)


In [ ]:
'''EPOCHS = 2

# -------------------------
# Trainer
# -------------------------
args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    learning_rate=LR,
    per_device_train_batch_size=BATCH_SIZE,
    num_train_epochs=EPOCHS,
    weight_decay=0.01,
    fp16=torch.cuda.is_available(),
    logging_steps=100,
    save_strategy="no",   # don’t save checkpoints
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    tokenizer=tokenizer,
    data_collator=collator
)

trainer.train()
'''

In [ ]:
import numpy as np
import torch

# ---------- choose one or many templates ----------
TEMPLATE_IDS = [2]           # start with one (e.g., 2). To ensemble, try [1,2,3,5]

@torch.no_grad()
def predict_logits(texts, model, tokenizer, batch_size=32, max_length=384):
    model.eval()
    out = []
    for i in range(0, len(texts), batch_size):
        enc = tokenizer(
            texts[i:i+batch_size],
            truncation=True,
            padding=True,
            max_length=max_length,
            return_tensors="pt"
        )
        enc = {k: v.to(model.device) for k, v in enc.items()}
        logits = model(**enc).logits
        out.append(logits.detach().cpu().numpy())
    return np.concatenate(out, axis=0)

# ---------- build predictions with A/B-swap TTA and logit averaging ----------
logits_sum = None
for tid in TEMPLATE_IDS:
    # normal order
    normal_texts = [
        format_input(r.prompt, r.response_a, r.response_b, tid)
        for _, r in test_df.iterrows()
    ]
    # swapped order
    swapped_texts = [
        format_input(r.prompt, r.response_b, r.response_a, tid)
        for _, r in test_df.iterrows()
    ]

    l1 = predict_logits(normal_texts, model, tokenizer, batch_size=32, max_length=MAX_LENGTH)
    l2 = predict_logits(swapped_texts, model, tokenizer, batch_size=32, max_length=MAX_LENGTH)

    # flip back A/B columns on the swapped logits (class order: [A, B, Tie] -> swap 0 and 1)
    l2[:, [0, 1]] = l2[:, [1, 0]]

    # average normal + swapped for this template (still in logit space)
    l_tpl = (l1 + l2) / 2.0

    logits_sum = l_tpl if logits_sum is None else (logits_sum + l_tpl)

# if multiple templates, average across them (still logits)
logits_avg = logits_sum / len(TEMPLATE_IDS)

# convert logits → probabilities once at the end
final_probs = torch.softmax(torch.tensor(logits_avg, dtype=torch.float32), dim=-1).numpy()

# ---------- write submission ----------
sub = pd.DataFrame({
    "id": test_df["id"],
    "winner_model_a": final_probs[:, 0],
    "winner_model_b": final_probs[:, 1],
    "winner_tie":     final_probs[:, 2],
})
sub.to_csv("/kaggle/working/submission.csv", index=False)
print("✅ submission.csv ready with TTA + logit averaging")
